# FP8 PS2: Auctions with Risk Reserves for Scarce AI Research
Qianshuo (Aaron) Wang and Zhenning Wang. Synthetic comparison, not a multi-slot equilibrium.
Run all cells. N=20, K=4, M=100, r=100ES, threshold=60, R=1, b=0.8v, lambda=1; seed 20260926; 1,000 independent batches, not dynamic rounds. The code runs 36 sensitivity scenarios. No external packages or downloads.


**Cell 1 — Setup and file paths**

In [ ]:
from pathlib import Path
import tempfile
import subprocess
import sys

# Create a temporary working directory
work = Path(tempfile.mkdtemp(prefix="ps2_"))

print("Working directory:", work)

Working directory: /tmp/ps2_eawiv18j


**Cell 2 — The core simulation model**

In [ ]:
simulation_code = r'''
"""FP8 PS2: paired, one-shot allocation experiment (Python standard library)."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import platform
import random
import statistics as st
from dataclasses import asdict, dataclass
from pathlib import Path

SEED = 20260926
BATCHES = 1000


@dataclass(frozen=True)
class Request:
    pm: int
    value: float
    exposure: float
    severity: float
    arrival: int
    shading: float = 0.8
    budget: float = 100.0

    @property
    def risk(self):
        return 100 * self.exposure * self.severity

    @property
    def bid(self):
        return min(self.budget, self.shading * self.value)


def generate_batches(
    seed=SEED,
    count=BATCHES,
    alignment="independent",
    heterogeneous=False
):
    """
    Generate synthetic batches of 20 PM requests.

    Baseline:
    - values ~ U[0,100]
    - exposure, severity ~ U[0,1]
    - bid = 0.8 * value

    Alternative value-risk alignments and heterogeneous shading
    are used only for sensitivity analysis.
    """
    rng = random.Random(seed)

    for _ in range(count):
        values = [100 * rng.random() for _ in range(20)]
        exposures = [rng.random() for _ in range(20)]
        severities = [rng.random() for _ in range(20)]

        arrival = list(range(20))
        rng.shuffle(arrival)

        shade = [0.5 + 0.5 * rng.random() for _ in range(20)]

        if alignment != "independent":
            rank = sorted(
                range(20),
                key=lambda j: (exposures[j] * severities[j], j)
            )

            assigned = sorted(
                values,
                reverse=(alignment == "opposed")
            )

            values = dict(zip(rank, assigned))

        yield [
            Request(
                j + 1,
                values[j],
                exposures[j],
                severities[j],
                arrival[j],
                shade[j] if heterogeneous else 0.8
            )
            for j in range(20)
        ]
'''

**Cell 3 — Allocation mechanisms and evaluation metrics**

In [ ]:
simulation_code += r'''


def allocate(
    requests,
    mechanism,
    slots=4,
    reserve=1,
    threshold=60
):
    """
    Apply one of three allocation mechanisms:
    pure first-price auction, hybrid reserve mechanism, or FCFS.
    """

    if mechanism not in ("pure", "hybrid", "fcfs"):
        raise ValueError("unknown mechanism")

    if not 0 <= reserve <= slots or slots < 0:
        raise ValueError("invalid reserve/slots")

    if len({r.pm for r in requests}) != len(requests):
        raise ValueError("duplicate PM")

    if any(
        not (
            0 <= r.value <= 100
            and 0 <= r.exposure <= 1
            and 0 <= r.severity <= 1
            and 0 <= r.shading <= 1
            and r.budget >= 0
        )
        for r in requests
    ):
        raise ValueError("invalid request")

    winners = {}

    # 1. First-Come, First-Served
    if mechanism == "fcfs":
        return {
            r.pm: 0.0
            for r in sorted(
                requests,
                key=lambda r: (r.arrival, r.pm)
            )[:slots]
        }

    # 2. Hybrid: reserve high-risk requests first
    if mechanism == "hybrid":
        eligible = sorted(
            (r for r in requests if r.risk >= threshold),
            key=lambda r: (-r.risk, r.pm)
        )

        winners = {
            r.pm: 0.0
            for r in eligible[:reserve]
        }

    # 3. Auction remaining slots
    remaining = sorted(
        (r for r in requests if r.pm not in winners),
        key=lambda r: (-r.bid, r.pm)
    )

    winners.update({
        r.pm: r.bid
        for r in remaining[:slots - len(winners)]
    })

    return winners


def metrics(
    requests,
    allocation,
    threshold=60,
    weight=1.0
):
    """Calculate value, risk coverage, welfare, payment and access."""

    chosen = [
        r for r in requests
        if r.pm in allocation
    ]

    value = sum(r.value for r in chosen)
    risk = sum(r.risk for r in chosen)
    payment = sum(allocation.values())

    optimum = sum(
        sorted(
            (r.value for r in requests),
            reverse=True
        )[:4]
    )

    eligible = sum(
        r.risk >= threshold
        for r in requests
    )

    return dict(
        V=value,
        C=risk,
        W=value + weight * risk,
        payment=payment,
        utility=value - payment,
        efficiency=value / optimum if optimum else None,
        eligible_access=(
            sum(r.risk >= threshold for r in chosen) / eligible
            if eligible
            else None
        )
    )
'''

**Cell 4 — Main simulation and sensitivity analysis**

In [ ]:
simulation_code += r'''


def write_csv(path, rows):
    rows = list(rows)

    with path.open(
        "w",
        newline="",
        encoding="utf-8"
    ) as handle:

        writer = csv.DictWriter(
            handle,
            fieldnames=list(rows[0])
        )

        writer.writeheader()
        writer.writerows(rows)


def mean(values):
    values = [
        v for v in values
        if v is not None
    ]

    return st.mean(values) if values else None


def run(
    out="results",
    seed=SEED,
    count=BATCHES
):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)

    # Generate matched synthetic batches
    batches = list(
        generate_batches(seed, count)
    )

    records = []
    individuals = []

    # Save one example batch
    demo = batches[0]

    write_csv(
        out / "demo_inputs.csv",
        [
            dict(
                **asdict(r),
                risk=r.risk,
                bid=r.bid
            )
            for r in demo
        ]
    )

    # --------------------------------------------------
    # Primary mechanism comparison
    # --------------------------------------------------

    for batch, requests in enumerate(batches, 1):

        for mechanism in (
            "pure",
            "hybrid",
            "fcfs"
        ):

            allocation = allocate(
                requests,
                mechanism
            )

            records.append(
                dict(
                    batch=batch,
                    mechanism=mechanism,
                    winners=";".join(
                        map(str, sorted(allocation))
                    ),
                    **metrics(
                        requests,
                        allocation
                    )
                )
            )

            # Save individual outcomes for the first batch
            if batch == 1:

                for r in requests:

                    payment = allocation.get(
                        r.pm,
                        0
                    )

                    allocated = int(
                        r.pm in allocation
                    )

                    individuals.append(
                        dict(
                            mechanism=mechanism,
                            pm=r.pm,
                            value=r.value,
                            risk=r.risk,
                            bid=r.bid,
                            allocated=allocated,
                            payment=payment,
                            utility=(
                                allocated * r.value
                                - payment
                            ),
                            credits_left=(
                                r.budget
                                - payment
                            )
                        )
                    )

    write_csv(
        out / "demo_allocations.csv",
        individuals
    )

    write_csv(
        out / "primary_batches.csv",
        records
    )

    # --------------------------------------------------
    # Mean outcomes by mechanism
    # --------------------------------------------------

    summary = []

    for mechanism in (
        "pure",
        "hybrid",
        "fcfs"
    ):

        subset = [
            r for r in records
            if r["mechanism"] == mechanism
        ]

        summary.append(
            dict(
                mechanism=mechanism,
                batches=count,
                **{
                    k: mean(
                        r[k]
                        for r in subset
                    )
                    for k in (
                        "V",
                        "C",
                        "W",
                        "payment",
                        "utility",
                        "efficiency",
                        "eligible_access"
                    )
                }
            )
        )

    write_csv(
        out / "primary_summary.csv",
        summary
    )

    # --------------------------------------------------
    # Sensitivity analysis
    # 3 alignments × 2 bid rules ×
    # 3 thresholds × 2 reserve sizes = 36 scenarios
    # --------------------------------------------------

    sensitivity = []

    for alignment in (
        "independent",
        "aligned",
        "opposed"
    ):

        for hetero in (
            False,
            True
        ):

            sample = list(
                generate_batches(
                    seed,
                    count,
                    alignment,
                    hetero
                )
            )

            for threshold in (
                40,
                60,
                80
            ):

                for reserve in (
                    1,
                    2
                ):

                    pairs = []

                    for requests in sample:

                        pure = metrics(
                            requests,
                            allocate(
                                requests,
                                "pure"
                            ),
                            threshold
                        )

                        hybrid = metrics(
                            requests,
                            allocate(
                                requests,
                                "hybrid",
                                reserve=reserve,
                                threshold=threshold
                            ),
                            threshold
                        )

                        pairs.append(
                            (
                                hybrid["V"] - pure["V"],
                                hybrid["C"] - pure["C"]
                            )
                        )

                    delta_V = mean(
                        p[0] for p in pairs
                    )

                    delta_C = mean(
                        p[1] for p in pairs
                    )

                    delta_W = [
                        v + c
                        for v, c in pairs
                    ]

                    mcse = (
                        st.stdev(delta_W)
                        / (count ** 0.5)
                        if count > 1
                        else 0
                    )

                    sensitivity.append(
                        dict(
                            alignment=alignment,
                            bid_rule=(
                                "heterogeneous"
                                if hetero
                                else "0.8v"
                            ),
                            threshold=threshold,
                            reserve=reserve,
                            delta_V=delta_V,
                            delta_C=delta_C,
                            delta_W_lambda0=delta_V,
                            delta_W_lambda05=(
                                delta_V
                                + 0.5 * delta_C
                            ),
                            delta_W_lambda1=(
                                delta_V
                                + delta_C
                            ),
                            delta_W_lambda2=(
                                delta_V
                                + 2 * delta_C
                            ),
                            MC_SE_lambda1=mcse,
                            fraction_strictly_better=mean(
                                x > 1e-10
                                for x in delta_W
                            ),
                            mean_break_even_lambda=(
                                -delta_V / delta_C
                                if delta_C > 1e-10
                                else None
                            )
                        )
                    )

    write_csv(
        out / "sensitivity.csv",
        sensitivity
    )

    # --------------------------------------------------
    # Reproducibility manifest
    # --------------------------------------------------

    source = Path(__file__)

    hashes = {
        p.name:
        hashlib.sha256(
            p.read_bytes()
        ).hexdigest()

        for p in sorted(
            out.glob("*.csv")
        )
    }

    manifest = dict(
        seed=seed,
        batches=count,
        python=platform.python_version(),
        N=20,
        K=4,
        credits=100,
        threshold=60,
        reserve=1,
        welfare_weight=1,
        primary_bid="0.8*v",
        risk="100*exposure*severity",
        distributions=(
            "independent U[0,100] value; "
            "U[0,1] exposure and severity; "
            "random arrival"
        ),
        simulation_sha256=(
            hashlib.sha256(
                source.read_bytes()
            ).hexdigest()
        ),
        outputs_sha256=hashes
    )

    (
        out / "fresh_run.json"
    ).write_text(
        json.dumps(
            manifest,
            indent=2
        ) + "\n"
    )

    print(
        json.dumps(
            summary,
            indent=2
        )
    )

    print(
        "Primary paired comparison:"
    )

    print(
        json.dumps(
            next(
                s
                for s in sensitivity
                if (
                    s["alignment"]
                    == "independent"
                    and s["bid_rule"]
                    == "0.8v"
                    and s["threshold"]
                    == 60
                    and s["reserve"]
                    == 1
                )
            ),
            indent=2
        )
    )

    return summary


if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--out",
        default="results"
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=SEED
    )

    parser.add_argument(
        "--batches",
        type=int,
        default=BATCHES
    )

    args = parser.parse_args()

    if args.batches < 1:
        parser.error(
            "batches must be positive"
        )

    run(
        args.out,
        args.seed,
        args.batches
    )
'''

In [ ]:
(work / "simulation.py").write_text(simulation_code)

print("simulation.py created.")

simulation.py created.


**Cell 5 — Unit tests**

In [ ]:
test_code = r'''
import unittest

from simulation import (
    Request,
    allocate,
    generate_batches,
    metrics
)


class MechanismTests(unittest.TestCase):

    def test_no_eligible_returns_reserve(self):
        requests = [
            Request(
                i,
                i * 10,
                0,
                1,
                i
            )
            for i in range(1, 7)
        ]

        self.assertEqual(
            allocate(
                requests,
                "pure"
            ),
            allocate(
                requests,
                "hybrid"
            )
        )


    def test_known_displacement_payments_and_welfare(self):

        requests = [
            Request(1, 100, .1, 1, 4),
            Request(2, 90, .2, 1, 3),
            Request(3, 80, .3, 1, 2),
            Request(4, 70, .4, 1, 1),
            Request(5, 10, .9, 1, 0)
        ]

        pure = allocate(
            requests,
            "pure"
        )

        hybrid = allocate(
            requests,
            "hybrid"
        )

        self.assertEqual(
            pure,
            {
                1: 80,
                2: 72,
                3: 64,
                4: 56
            }
        )

        self.assertEqual(
            hybrid,
            {
                5: 0,
                1: 80,
                2: 72,
                3: 64
            }
        )

        pure_metrics = metrics(
            requests,
            pure
        )

        hybrid_metrics = metrics(
            requests,
            hybrid
        )

        self.assertEqual(
            hybrid_metrics["W"]
            - pure_metrics["W"],
            -10
        )

        self.assertEqual(
            hybrid_metrics["utility"],
            64
        )


    def test_already_winner_no_allocation_change(self):

        requests = [
            Request(
                i,
                100 - i * 10,
                1 if i == 1 else 0,
                1,
                i
            )
            for i in range(1, 7)
        ]

        pure = allocate(
            requests,
            "pure"
        )

        hybrid = allocate(
            requests,
            "hybrid"
        )

        self.assertEqual(
            set(pure),
            set(hybrid)
        )

        self.assertEqual(
            hybrid[1],
            0
        )

        self.assertLess(
            sum(hybrid.values()),
            sum(pure.values())
        )


    def test_ties_boundary_and_fcfs(self):

        requests = [
            Request(
                i,
                50,
                .6,
                1,
                7 - i
            )
            for i in range(1, 7)
        ]

        self.assertEqual(
            allocate(
                requests,
                "hybrid"
            ),
            {
                1: 0,
                2: 40,
                3: 40,
                4: 40
            }
        )

        self.assertEqual(
            set(
                allocate(
                    requests,
                    "fcfs"
                )
            ),
            {
                3,
                4,
                5,
                6
            }
        )


    def test_budget_cap_and_invalid_duplicate(self):

        r = Request(
            1,
            100,
            1,
            1,
            0,
            budget=20
        )

        self.assertEqual(
            allocate(
                [r],
                "pure"
            ),
            {1: 20}
        )

        with self.assertRaises(
            ValueError
        ):
            allocate(
                [r, r],
                "pure"
            )


    def test_paired_invariants(self):

        for requests in generate_batches(
            count=100
        ):

            for mechanism in (
                "pure",
                "hybrid",
                "fcfs"
            ):

                allocation = allocate(
                    requests,
                    mechanism
                )

                outcome = metrics(
                    requests,
                    allocation
                )

                self.assertEqual(
                    len(allocation),
                    4
                )

                self.assertAlmostEqual(
                    outcome["utility"]
                    + outcome["payment"],
                    outcome["V"]
                )

                self.assertTrue(
                    0
                    <= outcome["efficiency"]
                    <= 1 + 1e-12
                )

                for r in requests:
                    self.assertTrue(
                        0
                        <= allocation.get(
                            r.pm,
                            0
                        )
                        <= r.budget
                    )

            pure_efficiency = metrics(
                requests,
                allocate(
                    requests,
                    "pure"
                )
            )["efficiency"]

            self.assertAlmostEqual(
                pure_efficiency,
                1
            )


if __name__ == "__main__":
    unittest.main()
'''

(work / "test_simulation.py").write_text(test_code)

print("test_simulation.py created.")

test_simulation.py created.


**Cell 6 — Run the tests**

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "-v",
        "test_simulation.py"
    ],
    cwd=work,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0


test_already_winner_no_allocation_change (test_simulation.MechanismTests.test_already_winner_no_allocation_change) ... ok
test_budget_cap_and_invalid_duplicate (test_simulation.MechanismTests.test_budget_cap_and_invalid_duplicate) ... ok
test_known_displacement_payments_and_welfare (test_simulation.MechanismTests.test_known_displacement_payments_and_welfare) ... ok
test_no_eligible_returns_reserve (test_simulation.MechanismTests.test_no_eligible_returns_reserve) ... ok
test_paired_invariants (test_simulation.MechanismTests.test_paired_invariants) ... ok
test_ties_boundary_and_fcfs (test_simulation.MechanismTests.test_ties_boundary_and_fcfs) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.033s

OK



**Cell 7 — Run the main simulation**

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "simulation.py",
        "--seed",
        "20260926",
        "--batches",
        "1000",
        "--out",
        "results"
    ],
    cwd=work,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

assert result.returncode == 0

[
  {
    "mechanism": "pure",
    "batches": 1000,
    "V": 353.076564567162,
    "C": 101.3511739173001,
    "W": 454.4277384844621,
    "payment": 282.4612516537296,
    "utility": 70.61531291343239,
    "efficiency": 1.0,
    "eligible_access": 0.20294351630867144
  },
  {
    "mechanism": "hybrid",
    "batches": 1000,
    "V": 326.257793396918,
    "C": 137.94518492411623,
    "W": 464.2029783210342,
    "payment": 226.97709330849966,
    "utility": 99.28070008841834,
    "efficiency": 0.9242441968495722,
    "eligible_access": 0.6670445505171042
  },
  {
    "mechanism": "fcfs",
    "batches": 1000,
    "V": 199.44904768342602,
    "C": 100.42880004917477,
    "W": 299.8778477326008,
    "payment": 0.0,
    "utility": 199.44904768342602,
    "efficiency": 0.5642785872523645,
    "eligible_access": 0.20930787589498806
  }
]
Primary paired comparison:
{
  "alignment": "independent",
  "bid_rule": "0.8v",
  "threshold": 60,
  "reserve": 1,
  "delta_V": -26.81877117024399,
  "delta_

## Interpretation
The reserve increases mean W by 9.78 at lambda=1 but reduces it by 8.52 at lambda=0.5. The mean break-even weight is 0.733. Only 43.8% of batches strictly improve at lambda=1. Perfect positive rank alignment gives no allocation benefit under common shading; negative rank alignment gives mean delta W=-3.24. These are synthetic policy-index results, not avoided financial losses.
Inspect `work / "results"` for all CSV outputs. See the repository README for parameters, algorithm, assumptions, limitations and AI disclosure. Human review and peer-play evidence remain separate.
